# Spreadsheets end to end

Loading an `.xlsx` with `read_excel`, working on it as a DataFrame, then
handing the numeric columns to the natural-language layer.

Requires `pip install -e ".[all,dev]"` from the repo root, and a provider key
in `examples/.env` (see the README's Setup section).

In [1]:
from dotenv import load_dotenv

load_dotenv(".env")

import numpy as np
import numpyai_dashboard as npi

## Loading

`read_excel` returns a `pandas.DataFrame`. Every column is kept, with its type
inferred by the Rust reader.

In [2]:
df = npi.read_excel("sample_sales.xlsx")
df.head()

,region,rep,product,order_date,units,unit_price,discount,closed,notes
0,EMEA,R. Ahmed,Basic,2023-01-01,21.0,19.99,NaN,False,renewal
1,APAC,R. Brown,Pro,2023-01-03,26.0,49.50,0.20,True,NaN
2,AMER,R. Chen,Enterprise,2023-01-05,5.0,199.00,0.25,True,NaN
3,LATAM,R. Duarte,Basic,2023-01-07,7.0,19.99,0.11,False,NaN
4,EMEA,R. Eriksen,Pro,2023-01-09,4.0,49.50,0.27,True,renewal


In [3]:
df.dtypes

region                   str
rep                      str
product                  str
order_date    datetime64[ms]
units                float64
unit_price           float64
discount             float64
closed                  bool
notes                    str
dtype: object

Text stays text, dates become `datetime64`, `TRUE`/`FALSE` become `bool`, and
blank cells become the null of whichever type the column is. Nothing is dropped.

In [4]:
df[["discount", "notes"]].isna().sum()

discount     14
notes       112
dtype: int64

### Options

Pick a sheet by name or index, skip the header row, or read only the first
`n_rows` when you just want to look at the shape of a large file.

In [5]:
npi.read_excel("sample_sales.xlsx", sheet="Sales", n_rows=5)

,region,rep,product,order_date,units,unit_price,discount,closed,notes
0,EMEA,R. Ahmed,Basic,2023-01-01,21.0,19.99,NaN,False,renewal
1,APAC,R. Brown,Pro,2023-01-03,26.0,49.50,0.20,True,NaN
2,AMER,R. Chen,Enterprise,2023-01-05,5.0,199.00,0.25,True,NaN
3,LATAM,R. Duarte,Basic,2023-01-07,7.0,19.99,0.11,False,NaN
4,EMEA,R. Eriksen,Pro,2023-01-09,4.0,49.50,0.27,True,renewal


## Working in pandas

Ordinary DataFrame work. Note `discount` has blanks, so fill them before
arithmetic.

In [6]:
df["revenue"] = df["units"] * df["unit_price"] * (1 - df["discount"].fillna(0))

df.groupby("region")["revenue"].sum().sort_values(ascending=False).round(2)

region
APAC     92209.21
EMEA     84916.77
LATAM    81800.69
AMER     75051.67
Name: revenue, dtype: float64

## Handing off to NumPy

`to_numpy()` on numeric columns is effectively free. Pass `columns=` so the
model knows what each column means rather than seeing bare indices.

In [7]:
cols = ["units", "unit_price", "discount", "revenue"]

arr = npi.array(df[cols].to_numpy(), columns=cols)
arr

numpyai_dashboard.array(shape=(150, 4), dtype=float64)

## Asking questions

Everything below needs a provider key. Generated code is syntax-checked and
independently judged before a result comes back.

In [8]:
arr.chat("What is the mean revenue?")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: What is the mean revenue?                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Judgment passed!

np.float64(2226.522308666667)

In [9]:
arr.chat("Correlation between units and revenue.")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: Correlation between units and revenue.                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Judgment passed!

np.float64(0.43744500530767105)

Missing values are visible to the model, so it can be asked about them directly.

In [10]:
arr.chat("How many rows have a missing discount?")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: How many rows have a missing discount?                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Judgment passed!

np.int64(14)

## Several arrays at once

`NumpyAISession` exposes each array to the model as `arr1`, `arr2`, ...

In [11]:
emea = df.loc[df["region"] == "EMEA", cols].to_numpy()
apac = df.loc[df["region"] == "APAC", cols].to_numpy()

sess = npi.NumpyAISession([emea, apac])
sess.chat("Compare the mean revenue of the two arrays.")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: Compare the mean revenue of the two arrays.                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

np.float64(-45.75129045438052)

## Diagnosis

Suggests analysis steps for the data rather than computing an answer.

In [12]:
diag = npi.Diagnosis(sess)
diag.steps(task="Give me 5 steps to analyse this sales data.")

────────────────────────────────────────────────── LLM Response ───────────────────────────────────────────────────

╭────────────────────────────────────────────── Data Analysis Steps ──────────────────────────────────────────────╮
│ [ "Step 1: Data Cleaning and Preparation: First, for both arr1 and arr2, identify and count all missing values  │
│ (NaNs) in each column. Specifically for the third column (index 2), which likely represents a discount rate,    │
│ replace these NaNs with 0, assuming that a missing discount value implies no discount was applied.              │
│ Additionally, identify any zero values in the 'total sales amount' column (index 3) to understand if there are  │
│ transactions with no recorded value. This step uses NumPy functions like isnan, count_nonzero, and nan_to_num.  │
│ The reasoning is that missing data can significantly bias or prevent subsequent analyses, and handling them     │
│ appropriately ensures data quality. Cleaned arrays are passed to the next step, and the count of NaNs and zeros │
│ helps diagnose data collection integrity.", "Step 2: Comprehensive Descriptive Statistics: For each cleaned     │
│ array, compute basic descriptive statistics including the mean, median, standard deviation, minimum, maximum,   │
│ and quartile (25th, 50th, 75th percentiles) for all numerical columns. Pay particular attention to the 'total   │
│ sales amount' (column 3), 'quantity' (column 0), and 'price' (column 1) to understand their central tendency,   │
│ dispersion, and range. NumPy functions such as mean, median, std, min, max, and percentile are relevant. This   │
│ step provides a foundational understanding of the data's characteristics, helping to identify typical values    │
│ and variability before deeper analysis. These statistics inform expectations for relationships explored in the  │
│ correlation analysis. Large standard deviations or significant differences between mean and median can indicate │
│ skewness or potential outliers.", "Step 3: Inter-variable Correlation Analysis: Calculate the Pearson           │
│ correlation matrix for the numerical columns (quantity, price, discount, total sales amount) within each        │
│ cleaned array. Specifically, examine the correlations between 'quantity' (column 0), 'price' (column 1),        │
│ 'discount' (column 2), and 'total sales amount' (column 3). The NumPy corrcoef function is ideal for this. The  │
│ reasoning is to quantify the linear relationships between different sales-related variables, for instance, how  │
│ discount application might influence total sales or how quantity sold scales with price. Understanding these    │
│ relationships helps in forming hypotheses for further investigation, and strong coefficients indicate           │
│ influential factors.", "Step 4: Sales Performance by Price Category: Identify the unique values present in the  │
│ 'price' column (column 1), treating these as distinct product categories or price tiers. Then, group the sales  │
│ records by these identified categories. For each category, compute aggregated metrics such as the total sales   │
│ amount, average quantity sold, and average discount applied. This involves using NumPy functions like unique    │
│ for identification and array indexing/masking combined with sum and mean for aggregation. This step is crucial  │
│ for assessing how different product price points contribute to overall sales and for identifying performance    │
│ trends specific to each category, which can then be used to compare performance between arrays. High sales in a │
│ category with low average quantity might suggest premium products.", "Step 5: Anomaly Detection in Total Sales: │
│ For each cleaned array, apply an outlier detection method to the 'total sales amount' (column 3). A robust      │
│ method is the Interquartile Range (IQR) method, where values falling below Q1 - 1.5IQR or above Q3 + 1.5IQR are │
│ identified as anomalies. Use NumPy's percentile to calculate Q1 and Q3, and array indexing/masking to filter.   │
│ The reasoning is to pinpoint unusually high or low sal

["**Step 1: Data Cleaning and Preparation:** First, for both `arr1` and `arr2`, identify and count all missing values (NaNs) in each column. Specifically for the third column (index 2), which likely represents a discount rate, replace these NaNs with 0, assuming that a missing discount value implies no discount was applied. Additionally, identify any zero values in the 'total sales amount' column (index 3) to understand if there are transactions with no recorded value. This step uses NumPy functions like `isnan`, `count_nonzero`, and `nan_to_num`. The reasoning is that missing data can significantly bias or prevent subsequent analyses, and handling them appropriately ensures data quality. Cleaned arrays are passed to the next step, and the count of NaNs and zeros helps diagnose data collection integrity.",
 "**Step 2: Comprehensive Descriptive Statistics:** For each cleaned array, compute basic descriptive statistics including the mean, median, standard deviation, minimum, maximum, and

## Verbose mode

`verbose=True` prints every intermediate step: the generated code, the
judgement, and any retries.

In [13]:
loud = npi.array(df[cols].to_numpy(), columns=cols, verbose=True)
loud.chat("Total revenue where units exceed 40.")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: Total revenue where units exceed 40.                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Attempt 1/3...

╭──────────────────────────────────────────────── Generated Code ─────────────────────────────────────────────────╮
│   1 arr_cleaned = np.nan_to_num(arr[:, 3])                                                                      │
│   2 output = np.sum(arr_cleaned[arr[:, 0] > 40])                                                                │
│   3 metadata = "Total revenue where units exceed 40"                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Executing generated code...

159278.66410000002

Total revenue where units exceed 40

╭─────────────────────────────────────────────────── Judgment ────────────────────────────────────────────────────╮
│ ✓ correctly interprets the query                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Judgment passed!

np.float64(159278.66410000002)